In [35]:
import re
import json
from pathlib import Path

In [36]:
def strip_comments(text: str) -> str:
    """Removes block comments first -- otherwise brackets/semicolons in
    JSDoc example code (e.g. an @media CSS example) would interfere with
    the subsequent brace-depth counting."""
    return re.sub(r'/\*.*?\*/', '', text, flags=re.DOTALL)


def find_interface_body(code: str, interface_name: str) -> str | None:
    """Finds the body of an 'interface Name<...>? (extends ...)? { ... }'
    block via brace-depth counting (not via a regex end-match), so that
    nested object types used as parameters (e.g. in slot methods) are
    recognized correctly.

    '[^{]*' instead of explicitly enumerating generics/'extends' clauses:
    some *Props interfaces (e.g. ButtonProps) are declared as
    'interface ButtonProps extends ButtonHTMLAttributes<...> { ... }'.
    A regex that only allows optional generics directly after the name
    would not find this block -- the function would then silently return
    None and the component would end up with props={} in the catalog,
    without any error being raised. '\\b' after the name prevents false
    matches such as 'ButtonPropsExtended'.
    """
    m = re.search(rf'\binterface\s+{re.escape(interface_name)}\b[^{{]*\{{', code)
    if not m:
        return None

    start = m.end()
    depth = 1
    i = start

    while i < len(code) and depth > 0:
        if code[i] == '{':
            depth += 1
        elif code[i] == '}':
            depth -= 1
        i += 1

    return code[start:i - 1]


def split_top_level_members(body: str) -> list[str]:
    """Splits an interface body at top-level semicolons.

    Counts ONLY {} and () as depth changes -- NOT '<'/'>': arrow-function
    types such as '(event: Event) => void' contain a '>', which would
    otherwise be falsely counted as a closing angle bracket of a generic
    and shift the entire subsequent count.
    """
    members, depth, current = [], 0, []

    for ch in body:
        if ch in '{(':
            depth += 1
        elif ch in '})':
            depth -= 1
        elif ch == ';' and depth == 0:
            members.append(''.join(current).strip())
            current = []
            continue

        current.append(ch)

    tail = ''.join(current).strip()
    if tail:
        members.append(tail)

    return [m for m in members if m]


def extract_member_name(member_text: str) -> str | None:
    m = re.match(r"""^(['"]?)([A-Za-z_][\w:-]*)\1""", member_text.strip())
    return m.group(2) if m else None


def extract_member_type(member_text: str) -> str | None:
    """Only meaningful for Props (Name: Type) -- Slots/Emits/PT-Keys are
    method signatures without a simple 'name: type' pattern."""
    m = re.match(r"""^['"]?[\w:-]+['"]?\??\s*:\s*(.+)$""", member_text.strip(), re.DOTALL)
    return re.sub(r'\s+', ' ', m.group(1)).strip() if m else None


def extract_component_api(dts_text: str, component_name: str) -> dict:
    """Extracts Props (name -> type string), slot names, PassThrough keys
    (first nesting level only) and emit names from a PrimeVue .d.ts file.
    """
    text = strip_comments(dts_text)

    props_body = find_interface_body(text, f'{component_name}Props')
    slots_body = find_interface_body(text, f'{component_name}Slots')
    pt_body    = find_interface_body(text, f'{component_name}PassThroughOptions')
    emits_body = find_interface_body(text, f'{component_name}EmitsOptions')

    props: dict[str, str | None] = {}
    if props_body:
        for member in split_top_level_members(props_body):
            name = extract_member_name(member)
            if name:
                props[name] = extract_member_type(member)

    slots   = [extract_member_name(m) for m in split_top_level_members(slots_body or '')]
    pt_keys = [extract_member_name(m) for m in split_top_level_members(pt_body or '')]
    emits   = [extract_member_name(m) for m in split_top_level_members(emits_body or '')]

    return {
        'props':   props,
        'slots':   [s for s in slots if s],
        'pt_keys': [k for k in pt_keys if k],
        'emits':   [e for e in emits if e],
    }


# Generic props that appear in practically every *Props interface (pt, dt,
# unstyled, ptOptions) -- handle separately for Prop-Validity Rate instead
# of duplicating them per component.
GENERIC_PROPS = {'pt', 'ptOptions', 'dt', 'unstyled'}


def is_static_bindable(type_str: str | None) -> bool:
    """Rough heuristic for the Binding-Correctness sub-metric: a type is
    statically bindable (prop="value" instead of :prop="value") if it is
    reducible to 'string' (including the HintedString<...> wrapper, which
    by definition is effectively a string type with autocomplete hints).
    Pure object/number/boolean/function types strictly require ':prop'.
    """
    if type_str is None:
        return False

    t = type_str.replace('| undefined', '').replace('|undefined', '').strip()

    if t == 'string':
        return True
    if t.startswith('HintedString<'):
        return True

    return False

In [37]:
def find_repo_root(marker: str = 'dataset', start: Path | None = None) -> Path:
    """Searches upward from the current working directory until a folder
    named 'marker' is found -- robust against the kernel's working directory
    not matching the notebook's own folder."""
    start = start or Path.cwd()

    for parent in [start, *start.parents]:
        if (parent / marker).is_dir():
            return parent

    raise FileNotFoundError(
        f"Could not find a folder named '{marker}' above {start}. "
        f'Current working directory: {Path.cwd()}'
    )


REPO_ROOT = find_repo_root()

# node_modules is located under dataset/storybook, not at the repo root
PRIMEVUE_NODE_MODULES = REPO_ROOT / 'dataset' / 'storybook' / 'node_modules' / 'primevue'

print(f'Repo root found:       {REPO_ROOT}')
print(f'PRIMEVUE_NODE_MODULES: {PRIMEVUE_NODE_MODULES}')
print(f'Exists:                {PRIMEVUE_NODE_MODULES.exists()}')

if not PRIMEVUE_NODE_MODULES.exists():
    raise FileNotFoundError(
        f'{PRIMEVUE_NODE_MODULES} does not exist -- check whether "npm install" '
        f'was run in dataset/storybook.'
    )

def build_dir_to_name_map(primevue_root: Path) -> dict[str, str]:
    """Reads primevue/index.d.ts (the root package file) and builds from it
    an authoritative mapping of directory name -> PascalCase component name.

    This file contains a line like the following for every component:
        export { default as MultiSelect } from 'primevue/multiselect';
    -- this is the source of truth straight from the package itself, more
    robust than deriving it via dir_name.capitalize() (which fails for
    multi-word names like 'multiselect' -> 'Multiselect' instead of
    'MultiSelect', since word boundaries cannot be reconstructed from a
    purely lowercase directory name) and more robust than a manually
    maintained override table.
    """
    index_dts = primevue_root / 'index.d.ts'

    if not index_dts.exists():
        print(f'WARNING: {index_dts} not found -- falling back to the capitalize() heuristic.')
        return {}

    text = index_dts.read_text(encoding='utf-8', errors='ignore')

    # Only the components' own default exports, no /style variants
    pairs = re.findall(r"export \{ default as (\w+) \} from 'primevue/([a-z0-9]+)';", text)

    mapping: dict[str, str] = {}
    for name, dir_name in pairs:
        mapping.setdefault(dir_name, name)  # first match wins (default export before /style)

    return mapping


DIR_TO_NAME = build_dir_to_name_map(PRIMEVUE_NODE_MODULES)
print(f'Mappings read from index.d.ts: {len(DIR_TO_NAME)}')

# Only used as a last-resort fallback, in case index.d.ts should fail to list a component
COMPONENT_NAME_OVERRIDES: dict[str, str] = {}


def component_name_from_dir(dir_name: str) -> str:
    if dir_name in DIR_TO_NAME:
        return DIR_TO_NAME[dir_name]

    if dir_name in COMPONENT_NAME_OVERRIDES:
        return COMPONENT_NAME_OVERRIDES[dir_name]

    guess = dir_name.capitalize()
    print(f'  WARNING: "{dir_name}" not found in index.d.ts, unreliable guess: "{guess}"')

    return guess


# Only these components are extracted (directory names under node_modules/primevue).
COMPONENTS = [
    # --- level 1 (15 PVK) ---

    # forms
    'checkbox',
    'inputnumber',
    'inputtext',
    'password',
    'radiobutton',
    'slider',
    'textarea',
    'toggleswitch',
    'iconfield', # special component from inputtext
    'inputicon', # special component from inputtext

    # buttons
    'button',

    # panel
    'divider',

    # misc
    'avatar',
    'badge',
    'overlaybadge', # special component from badge
    'progressbar',
    'skeleton',
    'tag',


    # --- level 2 (5 PVK) ---

    # panel
    'accordion',
    'accordionpanel', # special component from accordion
    'accordionheader', # special component from accordion
    'accordioncontent', # special component from accordion
    'card',
    'tabs',
    'tablist', # special component from tabs
    'tab', # special component from tabs
    'tabpanels', # special component from tabs
    'tabpanel', # special component from tabs

    # menu
    'breadcrumb',
    'menu',


    # --- level 3 (5 PVK) ---

    # form
    'datepicker',
    'select',

    # figma-data
    'datatable',
    'column', # special component from datatable

    # overlay
    'dialog',
    'popover',
]

API_CATALOG: dict[str, dict] = {}
skipped: list[str] = []
not_found: list[str] = []

for component_dir in COMPONENTS:
    dts_file = PRIMEVUE_NODE_MODULES / component_dir / 'index.d.ts'

    if not dts_file.exists():
        not_found.append(component_dir)
        print(f'  MISSING: {dts_file}')
        continue

    component_name = component_name_from_dir(component_dir)

    dts_text = dts_file.read_text(encoding='utf-8', errors='ignore')
    api = extract_component_api(dts_text, component_name)

    if not api['props'] and not api['slots']:
        skipped.append(component_dir)
        print(f'  {component_dir}: no *Props/*Slots interface found -- '
          f'check component_name "{component_name}"')
        continue

    API_CATALOG[component_name] = api

    print(f'{component_name:20s}  props={len(api["props"]):3d}  '
          f'slots={len(api["slots"]):2d}  pt_keys={len(api["pt_keys"]):2d}  emits={len(api["emits"]):2d}')

if not_found:
    print(f'\nNot found (directory does not exist under {PRIMEVUE_NODE_MODULES}): {not_found}')

if skipped:
    print(f'\nSkipped (no *Props/*Slots interface found): {skipped}')

print(f'\nCatalog complete: {len(API_CATALOG)} / {len(COMPONENTS)} requested components')

Repo root found:       C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo
PRIMEVUE_NODE_MODULES: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\node_modules\primevue
Exists:                True
Mappings read from index.d.ts: 136
Checkbox              props= 25  slots= 1  pt_keys= 5  emits= 6
InputNumber           props= 46  slots= 7  pt_keys= 9  emits= 5
InputText             props= 12  slots= 0  pt_keys= 2  emits= 2
Password              props= 42  slots= 8  pt_keys=13  emits= 3
RadioButton           props= 21  slots= 0  pt_keys= 5  emits= 5
Slider                props= 18  slots= 0  pt_keys= 6  emits= 4
Textarea              props= 13  slots= 0  pt_keys= 2  emits= 2
ToggleSwitch          props= 19  slots= 1  pt_keys= 5  emits= 5
IconField             props=  4  slots= 1  pt_keys= 2  emits= 0
InputIcon             props=  4  slots= 1  pt_keys= 2  emits= 0
Button                props= 27  slots= 3  pt_keys= 6  emits= 0
Divider               props=  7  slots= 1  

In [38]:
CATALOG_PATH = Path('primevue/component-types/primevue-api-types-v1.json')
CATALOG_PATH.write_text(json.dumps(API_CATALOG, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Saved: {CATALOG_PATH}  ({len(API_CATALOG)} components)')

Saved: primevue\component-types\primevue-api-types-v1.json  (36 components)
